# Scrapping User Dataset (Last.fm)


In [ ]:
import pandas as pd
import requests
import time
import re
import os

# ==========================================
# 1. KONFIGURASI
# ==========================================
LASTFM_API_KEY = 'b7a553d0be44366d0cc9a55d49d8f353'  

# Path dataset Spotify 
SPOTIFY_DATASET_PATH = '/content/spotify_top_songs_audio_features.csv'

# File output
OUTPUT_RAW_PATH = 'lastfm_interactions_raw.csv'
OUTPUT_CLEAN_PATH = 'lastfm_interactions_clean.csv'
VISITED_USERS_PATH = 'visited_users.txt'

# Target & Threshold
TARGET_INTERACTIONS = 600000
MIN_INTERACTIONS_USER = 10   # K-core: minimal interaksi per user
MIN_INTERACTIONS_ITEM = 10   # K-core: minimal interaksi per lagu
KCORE_ITERATIONS = 10        # Maksimal iterasi k-core filtering

# Seed users Last.fm (user aktif yang dikenal publik dan punya selera mainstream)
# Semakin banyak seed = semakin beragam jaringan snowball
SEED_USERS = [
    'rj', 'toke', 'mokele', 'carmona', 'nyarlathotep',
    'bumi_hills', 'Psjfpepper', 'Skylize', 'martin_hansen',
    'Maddieman', 'Joanasm', 'sprintcowboy', 'zsjp', 'DSJP',
    'rishavparijat', 'Atsjoo', 'Kisjansen', 'BelieveTomorrow',
    'Psjansen', 'Cornbolt'
]

# ==========================================
# 2. FUNGSI PEMBERSIHAN TEKS
# ==========================================
def clean_text(text):
    """Bersihkan teks untuk matching: lowercase, buang feat/remix info, buang simbol"""
    if pd.isna(text):
        return ""
    text = str(text).lower().strip()
    # Hapus info dalam kurung: (feat. ...), (remix), [deluxe], dll
    text = re.sub(r'\(.*?\)', '', text)
    text = re.sub(r'\[.*?\]', '', text)
    # Hapus semua karakter non-alfanumerik kecuali spasi
    text = re.sub(r'[^a-z0-9\s]', '', text)
    # Hapus spasi berlebih
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# ==========================================
# 3. LOAD DATASET SPOTIFY & BANGUN LOOKUP
# ==========================================
print("=" * 60)
print("MEMUAT DATASET SPOTIFY")
print("=" * 60)

df_spotify = pd.read_csv(SPOTIFY_DATASET_PATH)

# Kolom di dataset ini adalah 'id' (bukan 'track_id')
# dan 'artist_names' formatnya pakai KOMA sebagai pemisah multi-artist
# Contoh: "ZAYN, PARTYNEXTDOOR"

# Hapus duplikat berdasarkan id (Spotify track ID unik)
before = len(df_spotify)
df_spotify = df_spotify.drop_duplicates(subset='id', keep='first')
print(f"Duplikat ID dihapus: {before - len(df_spotify)}")

# Bangun beberapa variasi lookup untuk meningkatkan match rate
# Karena Last.fm dan Spotify sering beda format nama

spotify_lookup = {}

for _, row in df_spotify.iterrows():
    track_id = row['id']
    track_clean = clean_text(row['track_name'])
    artists_raw = str(row['artist_names']) if pd.notnull(row['artist_names']) else ""

    # Variasi 1: Artis pertama (sebelum koma pertama) + judul
    main_artist = clean_text(artists_raw.split(',')[0])
    key1 = f"{main_artist} - {track_clean}"
    if key1 not in spotify_lookup:
        spotify_lookup[key1] = track_id

    # Variasi 2: Setiap artis individual + judul (untuk kolaborasi)
    for artist in artists_raw.split(','):
        artist_clean = clean_text(artist)
        if artist_clean:
            key2 = f"{artist_clean} - {track_clean}"
            if key2 not in spotify_lookup:
                spotify_lookup[key2] = track_id

print(f"Total lagu unik: {df_spotify['id'].nunique()}")
print(f"Total variasi lookup key: {len(spotify_lookup)}")
print()

# ==========================================
# 4. FUNGSI API LAST.FM
# ==========================================
def get_user_top_tracks(username, limit=500, max_pages=2):
    """Ambil top tracks user dari Last.fm (paginasi)"""
    all_tracks = []
    url = "http://ws.audioscrobbler.com/2.0/"

    for page in range(1, max_pages + 1):
        params = {
            'method': 'user.gettoptracks',
            'user': username,
            'api_key': LASTFM_API_KEY,
            'format': 'json',
            'limit': limit,
            'page': page
        }
        try:
            res = requests.get(url, params=params, timeout=15)
            if res.status_code != 200:
                break
            data = res.json()

            # Cek apakah ada error dari API (user tidak ditemukan, dll)
            if 'error' in data:
                break

            tracks = data.get('toptracks', {}).get('track', [])
            if not tracks:
                break

            all_tracks.extend(tracks)
            time.sleep(0.25)  # Jeda antar halaman

        except (requests.exceptions.RequestException, ValueError):
            break

    return all_tracks


def get_user_friends(username, limit=50):
    """Ambil daftar teman user untuk snowball sampling"""
    url = "http://ws.audioscrobbler.com/2.0/"
    params = {
        'method': 'user.getfriends',
        'user': username,
        'api_key': LASTFM_API_KEY,
        'format': 'json',
        'limit': limit
    }
    try:
        res = requests.get(url, params=params, timeout=15)
        if res.status_code != 200:
            return []
        data = res.json()
        if 'error' in data:
            return []
        friends = data.get('friends', {}).get('user', [])
        return [f.get('name') for f in friends
                if isinstance(f, dict) and f.get('name')]
    except (requests.exceptions.RequestException, ValueError):
        return []

# ==========================================
# 5. RESUME: MUAT PROGRESS SEBELUMNYA
# ==========================================
interaction_data = []
visited_users = set()

# Muat data interaksi yang sudah tersimpan sebelumnya (jika ada)
if os.path.exists(OUTPUT_RAW_PATH):
    df_existing = pd.read_csv(OUTPUT_RAW_PATH)
    interaction_data = df_existing.to_dict('records')
    print(f"[RESUME] Memuat {len(interaction_data)} interaksi dari file sebelumnya.")

# Muat daftar user yang sudah pernah di-scrape (jika ada)
if os.path.exists(VISITED_USERS_PATH):
    with open(VISITED_USERS_PATH, 'r') as f:
        visited_users = set(line.strip() for line in f if line.strip())
    print(f"[RESUME] Memuat {len(visited_users)} user yang sudah dikunjungi.")

total_interactions = len(interaction_data)
users_processed = len(visited_users)

print(f"[RESUME] Mulai dari: {total_interactions} interaksi, {users_processed} user\n")

# ==========================================
# 6. SNOWBALL CRAWLING
# ==========================================
print("=" * 60)
print(f"MULAI CRAWLING — TARGET: {TARGET_INTERACTIONS} INTERAKSI")
print("=" * 60)

# Inisialisasi queue: seed users + teman-teman yang belum dikunjungi
user_queue = [u for u in SEED_USERS if u not in visited_users]
skip_count = 0
no_match_streak = 0

while user_queue and total_interactions < TARGET_INTERACTIONS:
    current_user = user_queue.pop(0)

    if current_user in visited_users:
        continue

    visited_users.add(current_user)
    users_processed += 1

    # --- Ambil top tracks ---
    tracks = get_user_top_tracks(current_user, limit=500, max_pages=2)

    if not tracks:
        print(f"  [{users_processed}] {current_user}: SKIP (tidak bisa diakses)")
        skip_count += 1
        time.sleep(0.5)

        # Tetap ambil teman meski track gagal
        friends = get_user_friends(current_user, limit=30)
        for friend in friends:
            if friend not in visited_users and friend not in user_queue:
                user_queue.append(friend)
        continue

    # --- Matching track ke dataset Spotify ---
    match_count = 0

    for track in tracks:
        lf_track = track.get('name', '')
        lf_artist = track.get('artist', {}).get('name', '')
        playcount = int(track.get('playcount', 0))

        # Skip jika playcount 0 (tidak ada interaksi nyata)
        if playcount <= 0:
            continue

        # Buat key untuk matching
        match_key = f"{clean_text(lf_artist)} - {clean_text(lf_track)}"

        if match_key in spotify_lookup:
            interaction_data.append({
                'user_id': current_user,
                'track_id': spotify_lookup[match_key],
                'playcount': playcount
            })
            match_count += 1
            total_interactions += 1

    # Tracking progres
    if match_count > 0:
        no_match_streak = 0
    else:
        no_match_streak += 1

    progress_pct = (total_interactions / TARGET_INTERACTIONS) * 100
    print(f"  [{users_processed}] {current_user}: "
          f"+{match_count} match | "
          f"Total: {total_interactions:,}/{TARGET_INTERACTIONS:,} ({progress_pct:.1f}%) | "
          f"Queue: {len(user_queue)}")

    # --- Snowball: ambil teman ---
    friends = get_user_friends(current_user, limit=30)
    new_friends = [f for f in friends
                   if f not in visited_users and f not in user_queue]
    user_queue.extend(new_friends)

    # --- Auto-save setiap 25 user ---
    if users_processed % 25 == 0:
        pd.DataFrame(interaction_data).to_csv(OUTPUT_RAW_PATH, index=False)
        with open(VISITED_USERS_PATH, 'w') as f:
            f.write('\n'.join(visited_users))
        print(f"  >>> AUTO-SAVE: {total_interactions:,} interaksi, "
              f"{len(visited_users)} user tersimpan.")

    # Jeda untuk menghindari rate limit
    time.sleep(0.5)

    # Warning jika terlalu banyak user tanpa match
    if no_match_streak >= 20:
        print(f"\n  [WARNING] {no_match_streak} user berturut-turut tanpa match.")
        print(f"  Queue masih: {len(user_queue)} user. Melanjutkan...\n")
        no_match_streak = 0

# ==========================================
# 7. SIMPAN HASIL MENTAH
# ==========================================
print("\n" + "=" * 60)
print("CRAWLING SELESAI")
print("=" * 60)

df_raw = pd.DataFrame(interaction_data)
df_raw.to_csv(OUTPUT_RAW_PATH, index=False)

with open(VISITED_USERS_PATH, 'w') as f:
    f.write('\n'.join(visited_users))

print(f"Total interaksi mentah  : {len(df_raw):,}")
print(f"Total user unik         : {df_raw['user_id'].nunique() if len(df_raw) > 0 else 0}")
print(f"Total lagu unik         : {df_raw['track_id'].nunique() if len(df_raw) > 0 else 0}")
print(f"User yang dikunjungi    : {len(visited_users)}")
print(f"User yang di-skip       : {skip_count}")
print(f"\nFile tersimpan: {OUTPUT_RAW_PATH}")

# ==========================================
# 8. PREVIEW & STATISTIK
# ==========================================
if len(df_raw) > 0:
    print("\n" + "=" * 60)
    print("STATISTIK DATA MENTAH")
    print("=" * 60)

    user_counts = df_raw.groupby('user_id').size()
    item_counts = df_raw.groupby('track_id').size()

    print(f"\nInteraksi per user:")
    print(f"  Min: {user_counts.min()}, Max: {user_counts.max()}, "
          f"Median: {user_counts.median():.0f}, Mean: {user_counts.mean():.1f}")

    print(f"\nInteraksi per lagu:")
    print(f"  Min: {item_counts.min()}, Max: {item_counts.max()}, "
          f"Median: {item_counts.median():.0f}, Mean: {item_counts.mean():.1f}")

    # Preview berapa yang akan tersisa setelah k-core filtering
    print(f"\n--- Preview K-Core Filtering (threshold={MIN_INTERACTIONS_USER}/{MIN_INTERACTIONS_ITEM}) ---")
    df_temp = df_raw.copy()
    for i in range(KCORE_ITERATIONS):
        before_len = len(df_temp)
        user_counts_temp = df_temp.groupby('user_id').size()
        valid_users = user_counts_temp[user_counts_temp >= MIN_INTERACTIONS_USER].index
        df_temp = df_temp[df_temp['user_id'].isin(valid_users)]

        item_counts_temp = df_temp.groupby('track_id').size()
        valid_items = item_counts_temp[item_counts_temp >= MIN_INTERACTIONS_ITEM].index
        df_temp = df_temp[df_temp['track_id'].isin(valid_items)]

        if len(df_temp) == before_len:
            print(f"  K-core stabil di iterasi {i+1}")
            break

    print(f"  Setelah filtering: {len(df_temp):,} interaksi, "
          f"{df_temp['user_id'].nunique()} user, "
          f"{df_temp['track_id'].nunique()} lagu")
    print(f"\n  Jika terlalu sedikit, turunkan threshold ke 15 atau 10.")


MEMUAT DATASET SPOTIFY
Duplikat ID dihapus: 0
Total lagu unik: 6513
Total variasi lookup key: 8955

[RESUME] Mulai dari: 0 interaksi, 0 user

MULAI CRAWLING — TARGET: 600000 INTERAKSI
  [1] rj: +43 match | Total: 43/600,000 (0.0%) | Queue: 19
  [2] toke: +1 match | Total: 44/600,000 (0.0%) | Queue: 48
  [3] mokele: +0 match | Total: 44/600,000 (0.0%) | Queue: 48
  [4] carmona: +0 match | Total: 44/600,000 (0.0%) | Queue: 75
  [5] nyarlathotep: +0 match | Total: 44/600,000 (0.0%) | Queue: 74
  [6] bumi_hills: SKIP (tidak bisa diakses)
  [7] Psjfpepper: SKIP (tidak bisa diakses)
  [8] Skylize: SKIP (tidak bisa diakses)
  [9] martin_hansen: SKIP (tidak bisa diakses)
  [10] Maddieman: SKIP (tidak bisa diakses)
  [11] Joanasm: +0 match | Total: 44/600,000 (0.0%) | Queue: 98
  [12] sprintcowboy: SKIP (tidak bisa diakses)
  [13] zsjp: +0 match | Total: 44/600,000 (0.0%) | Queue: 96
  [14] DSJP: SKIP (tidak bisa diakses)
  [15] rishavparijat: SKIP (tidak bisa diakses)
  [16] Atsjoo: +10 match 